## Etapa 1 — Inventário e Compreensão dos Dados

## Objetivo

## Compreender a estrutura da base RAIS e identificar os relacionamentos necessários para a modelagem analítica.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

arquivo = "../Bases/rais/rais_2020.csv" 


## Etapa 1.1 — Inspeção Inicial da Estrutura

## Objetivo

## Realizar uma leitura exploratória da base para identificar as colunas disponíveis e compreender a estrutura geral dos dados.


In [2]:
df = pd.read_csv(arquivo, nrows=5)

df.head()

df.info()


for i, coluna in enumerate(df.columns, start=1):
    print(f"{i}. {coluna}")



<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mun_trab                5 non-null      int64  
 1   ano                     5 non-null      int64  
 2   mes_admissao            5 non-null      int64  
 3   mes_desligamento        5 non-null      str    
 4   vinculo_ativo_31_12     5 non-null      int64  
 5   vl_remun_media_nom      5 non-null      float64
 6   cnae_2_0_classe         5 non-null      int64  
 7   sexo_trabalhador        5 non-null      int64  
 8   raca_cor                5 non-null      int64  
 9   tempo_emprego           5 non-null      str    
 10  escolaridade_apos_2005  5 non-null      int64  
 11  faixa_etaria            5 non-null      int64  
dtypes: float64(1), int64(9), str(2)
memory usage: 612.0 bytes
1. mun_trab
2. ano
3. mes_admissao
4. mes_desligamento
5. vinculo_ativo_31_12
6. vl_remun_media_nom
7. cn

## Etapa 1.2 — Análise do Dicionário de Dados

## Objetivo

## Consultar o dicionário de dados da RAIS para compreender o significado das variáveis presentes na base principal e apoiar as etapas de tratamento e modelagem.



In [3]:
aux = pd.read_excel("../Bases/rais_auxiliar.xls")

aux.head()



aux.info()

<class 'pandas.DataFrame'>
RangeIndex: 352 entries, 0 to 351
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Nome                   60 non-null     str    
 1   Descricao da Variável  59 non-null     str    
 2   Tamanho(em bytes)      60 non-null     float64
 3   Categorias             352 non-null    object 
 4   Valor na Fonte         340 non-null    object 
 5   Unnamed: 5             29 non-null     str    
dtypes: float64(1), object(2), str(3)
memory usage: 16.6+ KB


## Observação:
 
## Foi identificada uma coluna denominada `Unnamed: 5`, indicando uma possível coluna vazia ou artefato de formatação do arquivo Excel. Como essa coluna não possui relevância para a análise, ela não será utilizada nas etapas posteriores.

## Etapa 1.3 — Identificação das Tabelas Auxiliares
 
## Objetivo
 
## Mapear as planilhas disponíveis no arquivo auxiliar da RAIS para identificar as tabelas de apoio que serão utilizadas durante o tratamento e enriquecimento dos dados.


In [4]:
xls = pd.ExcelFile("../Bases/rais_auxiliar.xls")

for aba in xls.sheet_names:
    print(aba)


RAISD - layout
municipio
subclasse 2.0
classe 1.0 ou 95
FAIXAS
escolaridade ou g instruçao
Distrito SP
mes adm ou desl
REG ADM DF


## Etapa 1.4 — Seleção das Tabelas de Apoio

## Objetivo

## Identificar e carregar as tabelas auxiliares que serão utilizadas na construção das dimensões e no enriquecimento dos dados da RAIS.


In [5]:
municipio_regiao = pd.read_excel(
    "../Bases/municipio_regiao.ods",
    engine="odf",
    skiprows=6
)

municipio_regiao.head()



,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município,Unnamed: 9
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,NaN
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,379,1100379,Alto Alegre dos Parecis,NaN
2,11,Rondônia,1101,Porto Velho,110002,Ariquemes,403,1100403,Alto Paraíso,NaN
3,11,Rondônia,1102,Ji-Paraná,110004,Ji-Paraná,346,1100346,Alvorada D'Oeste,NaN
4,11,Rondônia,1101,Porto Velho,110002,Ariquemes,23,1100023,Ariquemes,NaN


## Etapa 1.4.1 — Tabela Município x Região

## Objetivo

## Carregar a tabela de correspondência entre municípios e regiões geográficas para posterior enriquecimento da base principal.



In [6]:

municipio_regiao.columns



Index(['UF', 'Nome_UF', 'Região Geográfica Intermediária',
       'Nome Região Geográfica Intermediária', 'Região Geográfica Imediata',
       'Nome Região Geográfica Imediata', 'Município',
       'Código Município Completo', 'Nome_Município', 'Unnamed: 9'],
      dtype='str')

## Validação da Estrutura

## Objetivo

## Verificar a quantidade de registros, os tipos de dados e a completude das colunas da tabela de municípios.



In [7]:
municipio_regiao.info()


<class 'pandas.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 10 columns):
 #   Column                                Non-Null Count  Dtype
---  ------                                --------------  -----
 0   UF                                    5571 non-null   int64
 1   Nome_UF                               5571 non-null   str  
 2   Região Geográfica Intermediária       5571 non-null   int64
 3   Nome Região Geográfica Intermediária  5571 non-null   str  
 4   Região Geográfica Imediata            5571 non-null   int64
 5   Nome Região Geográfica Imediata       5571 non-null   str  
 6   Município                             5571 non-null   int64
 7   Código Município Completo             5571 non-null   int64
 8   Nome_Município                        5571 non-null   str  
 9   Unnamed: 9                            1 non-null      str  
dtypes: int64(5), str(5)
memory usage: 435.4 KB


## Validação de Valores Nulos

## Objetivo

## Verificar a existência de valores ausentes nas colunas da tabela município x região após a remoção da coluna auxiliar sem utilização analítica.



In [8]:
municipio_regiao.isna().sum()


UF                                         0
Nome_UF                                    0
Região Geográfica Intermediária            0
Nome Região Geográfica Intermediária       0
Região Geográfica Imediata                 0
Nome Região Geográfica Imediata            0
Município                                  0
Código Município Completo                  0
Nome_Município                             0
Unnamed: 9                              5570
dtype: int64

## Etapa 2 — Preparação e Tratamento dos Dados

## Objetivo 

## Validar a consistência das bases RAIS 2020, 2021 e 2022 para construção da tabela fato consolidada.

## Etapa 2.1 — Carregamento das Bases

## Objetivo

## Importar as bases RAIS dos anos de 2020, 2021 e 2022 para validação da estrutura e posterior consolidação dos dados.


In [9]:
rais_2020 = pd.read_csv("../Bases/rais/rais_2020.csv", low_memory=False)
rais_2021 = pd.read_csv("../Bases/rais/rais_2021.csv", low_memory=False)
rais_2022 = pd.read_csv("../Bases/rais/rais_2022.csv", low_memory=False)


## Etapa 2.2 — Validação da Estrutura

## Objetivo

## Verificar se as bases possuem estrutura compatível para consolidação e identificar possíveis divergências entre colunas e tipos de dados.


In [10]:
for ano, df in {
    "2020": rais_2020,
    "2021": rais_2021,
    "2022": rais_2022
}.items():

    print(f"\n{'='*20}")
    print(f"RAIS {ano}")
    print(f"{'='*20}")

    df.info()



RAIS 2020
<class 'pandas.DataFrame'>
RangeIndex: 2183080 entries, 0 to 2183079
Data columns (total 12 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   mun_trab                int64  
 1   ano                     int64  
 2   mes_admissao            int64  
 3   mes_desligamento        str    
 4   vinculo_ativo_31_12     int64  
 5   vl_remun_media_nom      float64
 6   cnae_2_0_classe         int64  
 7   sexo_trabalhador        int64  
 8   raca_cor                int64  
 9   tempo_emprego           str    
 10  escolaridade_apos_2005  int64  
 11  faixa_etaria            int64  
dtypes: float64(1), int64(9), str(2)
memory usage: 199.9 MB

RAIS 2021
<class 'pandas.DataFrame'>
RangeIndex: 2472398 entries, 0 to 2472397
Data columns (total 12 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   mun_trab                int64  
 1   ano                     int64  
 2   mes_admissao            int64  
 3   mes_de

## Etapa 2.3 — Investigação de Inconsistências

## Objetivo

## Identificar valores inconsistentes na coluna mes_desligamento antes da aplicação das regras de tratamento.


In [11]:

for ano, df in {
    "2020": rais_2020,
    "2021": rais_2021,
    "2022": rais_2022
}.items():

    print(f"\n=== {ano} ===")

    print(
        df["mes_desligamento"]
        .value_counts(dropna=False)
        .head(20)
    )



=== 2020 ===
mes_desligamento
{ñ    1474062
12      81339
03      68423
11      62255
10      61343
02      60841
05      59896
09      57145
04      55820
01      52485
08      51691
06      49679
07      48101
Name: count, dtype: int64

=== 2021 ===
mes_desligamento
{ñ    1536823
12     122177
02      88321
03      78603
08      77182
07      76054
09      75483
10      74950
05      72607
11      71465
06      69320
04      66861
01      62552
Name: count, dtype: int64

=== 2022 ===
mes_desligamento
{ñ    2337835
12     158520
03     130976
05     118604
08     118351
02     117498
07     113243
01     111833
10     111071
06     109279
04     109014
09     108076
11     102180
Name: count, dtype: int64


## Etapa 2.4 — Identificação dos Valores Válidos

## Objetivo
## Mapear os valores existentes na coluna mes_desligamento para apoiar a definição das regras de padronização.


In [12]:
for ano, df in {
    "2020": rais_2020,
    "2021": rais_2021,
    "2022": rais_2022
}.items():

    print(f"\n=== {ano} ===")

    print(
        sorted(
            df["mes_desligamento"]
            .dropna()
            .astype(str)
            .unique()
        )
    )



=== 2020 ===
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '{ñ']

=== 2021 ===
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '{ñ']

=== 2022 ===
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '{ñ']


## Etapa 2.5 — Padronização dos Dados

## Objetivo 
## Padronizar os valores da variável mes_desligamento, removendo espaços em branco e substituindo códigos inconsistentes por valores ausentes.


In [13]:
for df in [rais_2020, rais_2021, rais_2022]:

    df["mes_desligamento"] = (
        df["mes_desligamento"]
        .astype(str)
        .str.strip()
        .replace("{ñ", pd.NA)
    )

## Etapa 2.6 — Conversão de Tipo de Dados

## Objetivo 

## Converter a variável mes_desligamento para formato numérico, preservando valores ausentes para utilização nas etapas posteriores.


In [14]:

for df in [rais_2020, rais_2021, rais_2022]:

    df["mes_desligamento"] = (
        pd.to_numeric(
            df["mes_desligamento"],
            errors="coerce"
        )
        .astype("Int64")
    )


## Etapa 2.7 — Validação Pós-Padronização

## Objetivo

##  Confirmar que os valores inconsistentes foram tratados corretamente e que a variável está pronta para consolidação das bases.


### Etapa 2.7.1 — Verificação dos Valores Ausentes


In [15]:
for ano, df in {
    "2020": rais_2020,
    "2021": rais_2021,
    "2022": rais_2022
}.items():

    print(f"\n=== {ano} ===")

    print(df["mes_desligamento"].isna().sum())



=== 2020 ===
1474062

=== 2021 ===
1536823

=== 2022 ===
2337835


### Etapa 2.7.2 — Verificação do Tipo de Dado


In [16]:
for ano, df in {
    "2020": rais_2020,
    "2021": rais_2021,
    "2022": rais_2022
}.items():

    print(
        ano,
        df["mes_desligamento"].dtype
    )


2020 Int64
2021 Int64
2022 Int64


### Etapa 2.7.3 — Valores Finais


In [17]:

for ano, df in {
    "2020": rais_2020,
    "2021": rais_2021,
    "2022": rais_2022
}.items():

    print(f"\n=== {ano} ===")

    print(
        sorted(
            df["mes_desligamento"]
            .dropna()
            .unique()
        )
    )



=== 2020 ===
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

=== 2021 ===
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

=== 2022 ===
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


### Etapa 2.7.4 — Distribuição Final


In [18]:
for ano, df in {
    "2020": rais_2020,
    "2021": rais_2021,
    "2022": rais_2022
}.items():

    print(f"\n=== {ano} ===")

    print(
        df["mes_desligamento"]
        .value_counts(dropna=False)
        .head(20)
    )


=== 2020 ===
mes_desligamento
<NA>    1474062
12        81339
3         68423
11        62255
10        61343
2         60841
5         59896
9         57145
4         55820
1         52485
8         51691
6         49679
7         48101
Name: count, dtype: Int64

=== 2021 ===
mes_desligamento
<NA>    1536823
12       122177
2         88321
3         78603
8         77182
7         76054
9         75483
10        74950
5         72607
11        71465
6         69320
4         66861
1         62552
Name: count, dtype: Int64

=== 2022 ===
mes_desligamento
<NA>    2337835
12       158520
3        130976
5        118604
8        118351
2        117498
7        113243
1        111833
10       111071
6        109279
4        109014
9        108076
11       102180
Name: count, dtype: Int64



## Etapa 2.8 — Comparação da Estrutura das Bases

## Objetivo

## Verificar se as três bases possuem exatamente as mesmas colunas antes da consolidação dos dados.


In [19]:
cols_2020 = set(rais_2020.columns)
cols_2021 = set(rais_2021.columns)
cols_2022 = set(rais_2022.columns)

print("2020 x 2021:", cols_2020 == cols_2021)
print("2020 x 2022:", cols_2020 == cols_2022)
print("2021 x 2022:", cols_2021 == cols_2022)



2020 x 2021: True
2020 x 2022: True
2021 x 2022: True



## Etapa 2.9 — Comparação dos Tipos de Dados

## Objetivo

## Identificar possíveis diferenças nos tipos de dados entre as bases para evitar inconsistências durante a consolidação.


In [20]:
tipos = pd.DataFrame({
    "2020": rais_2020.dtypes,
    "2021": rais_2021.dtypes,
    "2022": rais_2022.dtypes
})

tipos

,2020,2021,2022
mun_trab,int64,int64,int64
ano,int64,int64,int64
mes_admissao,int64,int64,int64
mes_desligamento,Int64,Int64,Int64
vinculo_ativo_31_12,int64,int64,int64
vl_remun_media_nom,float64,float64,float64
cnae_2_0_classe,int64,int64,int64
sexo_trabalhador,int64,int64,int64
raca_cor,int64,int64,int64
tempo_emprego,str,str,str


## Etapa 2.10 — Análise de Valores Nulos

## Objetivo

## Avaliar a presença de valores ausentes nas variáveis das bases e identificar possíveis impactos nas análises posteriores.


In [21]:

def resumo_nulos(df):

    return pd.DataFrame({
        "nulos": df.isna().sum(),
        "%_nulos": round(df.isna().mean()*100, 2)
    }).sort_values("%_nulos", ascending=False)



nulos_2020 = resumo_nulos(rais_2020)
nulos_2021 = resumo_nulos(rais_2021)
nulos_2022 = resumo_nulos(rais_2022)



nulos_2020.head(20)


,nulos,%_nulos
mes_desligamento,1474062,67.52
mun_trab,0,0.00
ano,0,0.00
mes_admissao,0,0.00
vinculo_ativo_31_12,0,0.00
vl_remun_media_nom,0,0.00
cnae_2_0_classe,0,0.00
sexo_trabalhador,0,0.00
raca_cor,0,0.00
tempo_emprego,0,0.00



## Etapa 2.11 — Verificação de Registros Duplicados

## Objetivo 

## Identificar registros duplicados que possam comprometer a qualidade da base consolidada.



In [22]:
for ano, df in {
    "2020": rais_2020,
    "2021": rais_2021,
    "2022": rais_2022
}.items():

    print(f"\n=== {ano} ===")
    print("Duplicados:", df.duplicated().sum())



=== 2020 ===
Duplicados: 43753

=== 2021 ===
Duplicados: 56513

=== 2022 ===
Duplicados: 44889


## Etapa 2.12 — Análise de Cardinalidade

## Objetivo

## Avaliar a quantidade de valores distintos das principais variáveis categóricas utilizadas na modelagem dimensional.

In [23]:
campos = [
    "sexo_trabalhador",
    "raca_cor",
    "faixa_etaria",
    "escolaridade_apos_2005",
    "mun_trab",
    "cnae_2_0_classe"
]

for coluna in campos:

    print(f"\n{coluna}")

    print(
        rais_2022[coluna]
        .nunique()
    )



sexo_trabalhador
3

raca_cor
7

faixa_etaria
9

escolaridade_apos_2005
12

mun_trab
295

cnae_2_0_classe
636


## Etapa 2.13 — Validação de Cobertura da Dimensão Município

## Objetivo

## Verificar se todos os municípios presentes na RAIS possuem correspondência na tabela auxiliar de municípios e regiões.


In [24]:

municipio_regiao["mun_trab"] = (
    municipio_regiao["Código Município Completo"]
    .astype(str)
    .str[:-1]
    .astype(int)
)


municipios_rais = set(rais_2022["mun_trab"].unique())

municipios_dim = set(
    municipio_regiao["mun_trab"].unique()
)

faltantes = municipios_rais - municipios_dim

print("Municípios não encontrados:",
      len(faltantes))


Municípios não encontrados: 0


## Etapa 2.14 — Validação da Dimensão CNAE

## Objetivo

## Construir e validar a dimensão CNAE, garantindo cobertura dos códigos presentes na base RAIS e tratamento de códigos não classificados.


### Etapa 2.14.1 — Carregamento da Tabela CNAE

In [25]:
import os

os.listdir("../Bases")

cnae = pd.read_csv("../Bases/cnae.csv")

cnae.head(10)


,Classe,denominacao_classe,divisao,denominacao_divisao,Grupo,denominacao_grupo,Subclasse,denominacao_subclasse
0,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0170-9/00,Caça e serviços relacionados
1,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0163-6/00,Atividades de pós-colheita
2,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0162-8/99,Atividades de apoio à pecuária não especificadas anteriormente
3,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0162-8/03,Serviço de manejo de animais
4,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0162-8/02,Serviço de tosquiamento de ovinos
5,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0162-8/01,Serviço de inseminação artificial em animais
6,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0161-0/99,Atividades de apoio à agricultura não especificadas anteriormente
7,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0161-0/03,"Serviço de preparação de terreno, cultivo e colheita"
8,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0161-0/02,Serviço de poda de árvores para lavouras
9,01.11-3,Cultivo de cereais,1,"AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS",1.7,Caça e serviços relacionados,0161-0/01,Serviço de pulverização e controle de pragas agrícolas


### Etapa 2.14.2 — Verificação do Formato dos Códigos


In [26]:
rais_2022["cnae_2_0_classe"].sample(20)

cnae.columns


Index(['Classe', 'denominacao_classe', 'divisao', 'denominacao_divisao',
       'Grupo', 'denominacao_grupo', 'Subclasse', 'denominacao_subclasse'],
      dtype='str')

### Etapa 2.14.3 — Transformação dos Códigos CNAE


In [27]:
cnae["cnae_2_0_classe"] = (
    cnae["Classe"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace("-", "", regex=False)
    .astype(int)
)


cnae[
    ["Classe", "cnae_2_0_classe", "denominacao_classe"]
].head()



,Classe,cnae_2_0_classe,denominacao_classe
0,01.11-3,1113,Cultivo de cereais
1,01.11-3,1113,Cultivo de cereais
2,01.11-3,1113,Cultivo de cereais
3,01.11-3,1113,Cultivo de cereais
4,01.11-3,1113,Cultivo de cereais


### Etapa 2.14.4 — Criação da Dimensão CNAE


In [28]:
dim_cnae = (
    cnae[
        ["cnae_2_0_classe", "denominacao_classe"]
    ]
    .drop_duplicates()
    .sort_values("cnae_2_0_classe")
)


dim_cnae.shape

dim_cnae.head()

,cnae_2_0_classe,denominacao_classe
0,1113,Cultivo de cereais
574,1121,Cultivo de algodão herbáceo e de outras fibras de lavoura temporária
1148,1130,Cultivo de cana-de-açúcar
1722,1148,Cultivo de fumo
2296,1156,Cultivo de soja


### Etapa 2.14.5 — Validação de Cobertura

In [29]:

cnaes_rais = set(
    rais_2022["cnae_2_0_classe"].unique()
)

cnaes_dim = set(
    dim_cnae["cnae_2_0_classe"].unique()
)

faltantes = cnaes_rais - cnaes_dim

print("CNAEs não encontrados:", len(faltantes))

list(faltantes)[:20]

rais_2022[
    rais_2022["cnae_2_0_classe"].isin([977, 999])
]["cnae_2_0_classe"].value_counts()


rais_2022[
    rais_2022["cnae_2_0_classe"].isin([977, 999])
].head()


CNAEs não encontrados: 2


,mun_trab,ano,mes_admissao,mes_desligamento,vinculo_ativo_31_12,vl_remun_media_nom,cnae_2_0_classe,sexo_trabalhador,raca_cor,tempo_emprego,escolaridade_apos_2005,faixa_etaria
380092,420280,2022,11,<NA>,1,3485.37,977,1,2,"2,0",5,7
463478,420460,2022,4,<NA>,1,23224.64,977,1,2,"8,0",6,5
542467,420460,2022,5,<NA>,1,16138.77,977,1,9,"8,0",7,3
542474,421190,2022,6,<NA>,1,18588.15,977,1,2,"7,0",7,7
542491,420460,2022,5,<NA>,1,18968.71,977,1,9,"7,9",7,3


### Etapa 2.14.6 — Tratamento de CNAEs Não Classificados

In [30]:
dim_cnae.loc[len(dim_cnae)] = [
    977,
    "Não Classificado"
]

dim_cnae.loc[len(dim_cnae)] = [
    999,
    "Não Classificado"
]



### Etapa 2.14.7 — Validação Final

In [31]:
cnaes_dim = set(dim_cnae["cnae_2_0_classe"])

faltantes = cnaes_rais - cnaes_dim

print(len(faltantes))

0



## Etapa 3 — Consolidação das Bases RAIS
 
## Objetivo
 
## Unificar as bases RAIS 2020, 2021 e 2022 em uma única tabela para facilitar as análises, a modelagem dimensional e a construção dos dashboards no Power BI.
# 
# ---
# 
# ## Consolidação das Bases
# 
# As três bases foram consolidadas utilizando a função `concat()` do Pandas, preservando a estrutura original dos dados e adicionando todos os registros em uma única tabela.
# 
# ```python
# rais = pd.concat(
#     [
#         rais_2020,
#         rais_2021,
#         rais_2022
#     ],
#     ignore_index=True
# )
# ```
# 
# ---
# 
# ## Validação da Consolidação
# 
# ### Total de Registros
# 
# ```python
# print("Total de registros:", len(rais))
# ```
# 
# ### Distribuição por Ano
# 
# ```python
# rais["ano"].value_counts().sort_index()
# ```
# 
# ### Estrutura da Base Consolidada
# 
# ```python
# rais.info()
# ```
# 
# ---
# 


In [32]:


rais = pd.concat(
    [
        rais_2020,
        rais_2021,
        rais_2022
    ],
    ignore_index=True
)




print("Linhas:", len(rais))

print("\nRegistros por ano:")
print(rais["ano"].value_counts().sort_index())



Linhas: 8401958

Registros por ano:
ano
2020    2183080
2021    2472398
2022    3746480
Name: count, dtype: int64


## Etapa 3 - Para garantir que nada mudou após o concat.


In [33]:

rais.info()

<class 'pandas.DataFrame'>
RangeIndex: 8401958 entries, 0 to 8401957
Data columns (total 12 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   mun_trab                int64  
 1   ano                     int64  
 2   mes_admissao            int64  
 3   mes_desligamento        Int64  
 4   vinculo_ativo_31_12     int64  
 5   vl_remun_media_nom      float64
 6   cnae_2_0_classe         int64  
 7   sexo_trabalhador        int64  
 8   raca_cor                int64  
 9   tempo_emprego           str    
 10  escolaridade_apos_2005  int64  
 11  faixa_etaria            int64  
dtypes: Int64(1), float64(1), int64(9), str(1)
memory usage: 777.2 MB



## Etapa 4 — Construção do Modelo Dimensional

## Objetivo

## Construir as dimensões auxiliares e definir a tabela fato que será utilizada na modelagem do Power BI.

## A utilização de um modelo dimensional facilita a criação de medidas, melhora a performance dos dashboards e reduz a complexidade dos relacionamentos.

## Modelo Proposto

## Tabela Fato

### fato_vinculos

## Dimensões

### dim_municipio
### dim_cnae
### dim_sexo
### dim_raca
### dim_escolaridade
### dim_faixa_etaria

## Etapa 4.1 — Criação da Dimensão Município

## Objetivo

## Criar uma dimensão geográfica contendo apenas os municípios presentes na base consolidada da RAIS.

## Além da identificação do município, serão mantidas as informações de região geográfica imediata e intermediária para permitir análises territoriais.


In [34]:

dim_municipio = (
    municipio_regiao[
        municipio_regiao["mun_trab"].isin(
            rais["mun_trab"].unique()
        )
    ][
        [
            "mun_trab",
            "Nome_Município",
            "Nome Região Geográfica Imediata",
            "Nome Região Geográfica Intermediária"
        ]
    ]
    .drop_duplicates()
    .sort_values("mun_trab")
)

dim_municipio.head()

dim_municipio.shape

(295, 4)

## Etapa 4.2 — Visualização da Dimensão CNAE

## Objetivo

## Visuali

In [35]:

dim_cnae.shape

dim_cnae.head()

,cnae_2_0_classe,denominacao_classe
0,1113,Cultivo de cereais
574,1121,Cultivo de algodão herbáceo e de outras fibras de lavoura temporária
1148,1130,Cultivo de cana-de-açúcar
1722,1148,Cultivo de fumo
2296,1156,Cultivo de soja


## Etapa 4.3 — Criação da Dimensão Sexo
 
## Objetivo
 
## Traduzir os códigos de sexo presentes na RAIS para descrições compreensíveis.


In [36]:
dim_sexo = pd.DataFrame({
    "sexo_trabalhador": [1, 2, -1],
    "sexo": [
        "Masculino",
        "Feminino",
        "Ignorado"
    ]
})

dim_sexo


,sexo_trabalhador,sexo
0,1,Masculino
1,2,Feminino
2,-1,Ignorado



## Etapa 4.4 — Criação da Dimensão Raça/Cor
 
## Objetivo
 
## Traduzir os códigos de raça/cor presentes na RAIS para descrições compreensíveis.


In [37]:

dim_raca = pd.DataFrame({
    "raca_cor": [1, 2, 4, 6, 8, 9, -1],
    "descricao_raca": [
        "Indígena",
        "Branca",
        "Preta",
        "Amarela",
        "Parda",
        "Não Identificada",
        "Ignorado"
    ]
})

dim_raca

,raca_cor,descricao_raca
0,1,Indígena
1,2,Branca
2,4,Preta
3,6,Amarela
4,8,Parda
5,9,Não Identificada
6,-1,Ignorado



## Etapa 4.5 — Validação dos Códigos Categóricos
 
## Objetivo
 
## Verificar os códigos existentes na base consolidada antes da criação das dimensões de escolaridade e faixa etária.



In [38]:


sorted(
    rais["escolaridade_apos_2005"].unique()
)



sorted(
    rais["faixa_etaria"].unique()
)

[np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(99)]


## Etapa 4.6 — Criação da Dimensão Escolaridade
 
## Objetivo
 
## Traduzir os códigos de escolaridade da RAIS em descrições compreensíveis para análise.



In [39]:


dim_escolaridade = pd.DataFrame({
    "escolaridade_apos_2005": [
        1,2,3,4,5,6,7,8,9,10,11,99
    ],
    "escolaridade": [
        "Analfabeto",
        "Até 5ª Série Incompleta",
        "5ª Série Completa Fundamental",
        "6ª a 9ª Série Fundamental",
        "Fundamental Completo",
        "Médio Incompleto",
        "Médio Completo",
        "Superior Incompleto",
        "Superior Completo",
        "Mestrado",
        "Doutorado",
        "Não Identificado"
    ]
})

dim_escolaridade


,escolaridade_apos_2005,escolaridade
0,1,Analfabeto
1,2,Até 5ª Série Incompleta
2,3,5ª Série Completa Fundamental
3,4,6ª a 9ª Série Fundamental
4,5,Fundamental Completo
5,6,Médio Incompleto
6,7,Médio Completo
7,8,Superior Incompleto
8,9,Superior Completo
9,10,Mestrado



## Etapa 4.7 — Criação da Dimensão Faixa Etária
 
## Objetivo
 
## Traduzir os códigos de faixa etária da RAIS em categorias compreensíveis para análise demográfica.



In [ ]:
dim_faixa_etaria = pd.DataFrame({
    "faixa_etaria": [
        1,2,3,4,5,6,7,8,99
    ],
    "descricao_faixa_etaria": [
        "10 a 14 anos",
        "15 a 17 anos",
        "18 a 24 anos",
        "25 a 29 anos",
        "30 a 39 anos",
        "40 a 49 anos",
        "50 a 64 anos",
        "65 anos ou mais",
        "Não Identificada"
    ]
})

dim_faixa_etaria

,faixa_etaria,descricao_faixa_etaria
0,1,10 a 14 anos
1,2,15 a 17 anos
2,3,18 a 24 anos
3,4,25 a 29 anos
4,5,30 a 39 anos
5,6,40 a 49 anos
6,7,50 a 64 anos
7,8,65 anos ou mais
8,99,Não Identificada




## Etapa 4.8 — Validação do Modelo Dimensional
 
## Objetivo
 
## Verificar a estrutura das tabelas criadas antes da exportação dos datasets.


In [41]:
print("Fato:", rais.shape)
print("Município:", dim_municipio.shape)
print("CNAE:", dim_cnae.shape)
print("Sexo:", dim_sexo.shape)
print("Raça:", dim_raca.shape)
print("Escolaridade:", dim_escolaridade.shape)
print("Faixa Etária:", dim_faixa_etaria.shape)


Fato: (8401958, 12)
Município: (295, 4)
CNAE: (674, 2)
Sexo: (3, 2)
Raça: (7, 2)
Escolaridade: (12, 2)
Faixa Etária: (9, 2)



## Etapa 4.9 — Avaliação de Consumo de Memória

## Objetivo

## Verificar o consumo de memória da tabela fato consolidada antes da exportação dos arquivos finais.


In [42]:
rais.memory_usage(deep=True).sum() / 1024**2



np.float64(1169.8589324951172)


## Etapa 4.10 — Exportação do Modelo Dimensional

## Objetivo

## Exportar a tabela fato e as dimensões para arquivos independentes que serão utilizados na construção do modelo estrela no Power BI.


In [43]:

import os


rais.to_csv(
    "../assets/fato_vinculos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("fato_vinculos.csv exportado com sucesso.")



dim_municipio.to_csv(
    "../assets/dim_municipio.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_cnae.to_csv(
    "../assets/dim_cnae.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_sexo.to_csv(
    "../assets/dim_sexo.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_raca.to_csv(
    "../assets/dim_raca.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_escolaridade.to_csv(
    "../assets/dim_escolaridade.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_faixa_etaria.to_csv(
    "../assets/dim_faixa_etaria.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dimensões exportadas.")

## Validação dos Arquivos Exportados

## Confirmar a geração dos arquivos exportados e verificar seus respectivos tamanhos.

for arquivo in os.listdir("../assets"):

    caminho = os.path.join("../assets", arquivo)

    tamanho_mb = os.path.getsize(caminho) / 1024 / 1024

    print(f"{arquivo}: {tamanho_mb:.2f} MB")



fato_vinculos.csv exportado com sucesso.
Dimensões exportadas.
dim_cnae.csv: 0.04 MB
dim_escolaridade.csv: 0.00 MB
dim_faixa_etaria.csv: 0.00 MB
dim_municipio.csv: 0.01 MB
dim_raca.csv: 0.00 MB
dim_sexo.csv: 0.00 MB
fato_vinculos.csv: 410.09 MB
notebook.png: 0.19 MB




## Etapa 4.11 — Validação das Chaves do Modelo
 
## Objetivo
 
## Verificar a existência de valores nulos nas colunas utilizadas como chave de relacionamento entre a tabela fato e as dimensões.
 
## A ausência de valores nulos garante a integridade do modelo estrela e evita problemas de relacionamento durante a construção dos dashboards no Power BI.


In [44]:

print("Valores nulos nas chaves do modelo")

print("mun_trab:", rais["mun_trab"].isna().sum())
print("cnae_2_0_classe:", rais["cnae_2_0_classe"].isna().sum())
print("sexo_trabalhador:", rais["sexo_trabalhador"].isna().sum())
print("raca_cor:", rais["raca_cor"].isna().sum())
print("escolaridade_apos_2005:", rais["escolaridade_apos_2005"].isna().sum())
print("faixa_etaria:", rais["faixa_etaria"].isna().sum())



Valores nulos nas chaves do modelo
mun_trab: 0
cnae_2_0_classe: 0
sexo_trabalhador: 0
raca_cor: 0
escolaridade_apos_2005: 0
faixa_etaria: 0


## Conclusão da Etapa 4
 
## Foi construído um modelo dimensional composto por uma tabela fato e seis dimensões auxiliares.
 
## As tabelas foram validadas, exportadas para arquivos CSV independentes e preparadas para implementação do modelo estrela no Power BI.
 
## A validação final confirmou a ausência de valores nulos nas chaves de relacionamento, garantindo a integridade do modelo analítico.
 
## Status

## ✅ Concluído


## Etapa 5 — Preparação do Modelo no Power BI
 
## Objetivo
 
## Importar a tabela fato e as dimensões, configurar os relacionamentos e construir o modelo estrela que servirá de base para os dashboards do projeto.
